# Лабораторная работа N 2. Метод конечных элементов: температура стержня

**Курс:** Сеточные методы решения краевых задач, часть I.  
**Дата выдачи:** 2026.02.24  
**Срок сдачи:** 2026.03.03

Лабораторная работа для школьников 10-11 классов, знакомых с Python, производными и интегралами. Используются треугольные, то есть кусочно-линейные, базисные и весовые функции.

**Как работать:** читайте теорию, заполняйте функции вместо `pass`, запускайте ячейки сверху вниз. Импорты, параметры и построение графиков уже подготовлены. Незаполненная функция возвращает `None`; зависимый опыт будет пропущен с пояснением. После заполнения функций заново запустите весь блокнот. Подсказки с контрольными значениями не являются готовым кодом решения.

## 1. Условие задачи

### Стационарный одномерный теплообмен методом конечных элементов (до 10 баллов)

Имеется однородный стержень из нихрома с площадью поперечного сечения $1\,\text{мм}^2$ и длиной $L$. Его удельное электрическое сопротивление равно $\rho=1.2\,\dfrac{\text{Ом}\cdot\text{мм}^2}{\text{м}}$.

Теплопроводность левой половины стержня составляет $5\,\dfrac{\text{Вт}}{\text{м}\cdot\text{К}}$, а правой половины - $10\,\dfrac{\text{Вт}}{\text{м}\cdot\text{К}}$.

Граничные условия:

1. Один конец стержня помещён в лёд, а другой - в сосуд с кипящей водой.
2. Один конец стержня помещён в лёд, а другой конец теплоизолирован.

Стержень нагревается электрическим током $0.1\,\text{А}$, подаваемым на его концах. Длина стержня равна $L=0.5\,\text{м}$.

Найдите распределение температуры вдоль стержня.

Сформулируйте математическую модель.

Используйте метод конечных элементов (МКЭ, FEM) с линейными базисными функциями для решения уравнения модели с заданными граничными условиями.

1. Приведите минимальные теоретические сведения, обосновывающие формулировку математической модели.
2. Приведите аналитическое решение, если сможете.
3. Опишите шаги подготовки математической модели к программированию; сформулируйте алгоритм.
4. Реализуйте алгоритм на Python, используя доступные реализации решателей линейных систем, и проверьте свой решатель сравнением с аналитическими решениями, возможно, для упрощённых задач.
5. Численно решите предложенную задачу и представьте несколько графиков, демонстрирующих результаты.
6. Проведите параметрическое исследование, изменяя некоторые входные параметры и граничные условия в разумных диапазонах; визуализируйте результаты и сделайте выводы.

**Учебное пояснение.** Длину измеряем в метрах. В первом варианте принимаем $T(0)=0\,{}^{\circ}\mathrm{C}$, $T(L)=100\,{}^{\circ}\mathrm{C}$; во втором - $T(0)=0\,{}^{\circ}\mathrm{C}$, $T'(L)=0$. Не учитываем потери тепла через боковую поверхность. Коэффициенты считаем независимыми от температуры; при изменении параметров обсуждайте границы применимости этого предположения.

## 2. Теория и физическая модель

Сначала предположим, что поток энергии возникает из-за разности температур - это закон Фурье:
$$\Phi=-k\frac{\mathrm dT}{\mathrm dx}.$$
Здесь $\Phi$ - плотность теплового потока, то есть мощность, проходящая через единицу площади ($\mathrm{Вт/м^2}$); $k$ - коэффициент теплопроводности; $T$ - температура; $x$ - координата вдоль стержня. Если температура растёт вправо, поток направлен влево: именно это выражает знак минус.

Затем предположим, что изменение потока вдоль стержня вызвано источниками тепла. В стационарном состоянии тепло не накапливается. Для короткого участка длины $\mathrm dx$ выходящая мощность минус входящая равна выделившейся мощности:
$$S[\Phi(x+\mathrm dx)-\Phi(x)]=QS\,\mathrm dx.$$
Следовательно,
$$\frac{\mathrm d\Phi}{\mathrm dx}=Q.$$
При отсутствии источников поток постоянен.

Положительное $Q$ означает выделение тепла; поэтому выходящий поток больше входящего.

### Джоулево тепло и единицы

В нашем случае омические потери служат источником тепла:
$$P=I^2R,\qquad R=\frac{\rho L}{S},\qquad P=I^2\frac{\rho L}{S}.$$
Объём стержня $V=SL$, поэтому мощность на единицу объёма равна
$$Q=\frac PV=\frac{I^2\rho L}{SV}=\boxed{\frac{I^2\rho}{S^2}}.$$
Все величины должны быть в согласованных единицах:
$$S=10^{-6}\,\mathrm{м^2},\quad \rho=1.2\cdot10^{-6}\,\mathrm{Ом\cdot м},\quad I=0.1\,\mathrm A,$$
$$Q=1.2\cdot10^4\,\mathrm{Вт/м^3}.$$

Если распределение $k(x)$ известно, итоговое дифференциальное уравнение имеет вид
$$\boxed{\frac{\mathrm d}{\mathrm dx}\left(k(x)\frac{\mathrm dT}{\mathrm dx}\right)+Q=0}.$$
В этой задаче коэффициент кусочно-постоянный:
$$k(x)=\begin{cases}k_L,&x<L/2,\\k_R,&x\ge L/2.\end{cases}$$
Разность температур в градусах Цельсия равна разности в кельвинах, поэтому можно вычислять $T$ в градусах Цельсия.

## 3. Слабая формулировка

Приблизим точное решение $T(x)$ функцией $T_h(x)$ и определим **невязку** - остаток после подстановки приближённого решения в уравнение:
$$R(x)=\frac{\mathrm d}{\mathrm dx}\left(k\frac{\mathrm dT_h}{\mathrm dx}\right)+Q.$$
В методе Галёркина требуем ортогональности невязки каждой весовой (тестовой) функции $w_i$ из того же пространства приближения:
$$\int_0^L R(x)w_i(x)\,\mathrm dx=0,\qquad\forall i.$$
То есть средний остаток, измеренный каждой такой функцией, должен обращаться в ноль. Раскрывая выражение, получаем
$$\int_0^L\frac{\mathrm d}{\mathrm dx}\left(k\frac{\mathrm dT_h}{\mathrm dx}\right)w_i\,\mathrm dx+\int_0^L Qw_i\,\mathrm dx=0.$$

Применим интегрирование по частям к первому слагаемому:
$$\int_0^L\frac{\mathrm d}{\mathrm dx}\left(k\frac{\mathrm dT_h}{\mathrm dx}\right)w_i\,\mathrm dx=
\left[k\frac{\mathrm dT_h}{\mathrm dx}w_i\right]_0^L-
\int_0^L k\frac{\mathrm dT_h}{\mathrm dx}\frac{\mathrm dw_i}{\mathrm dx}\,\mathrm dx.$$
После подстановки и переноса слагаемых получаем слабую форму:
$$\boxed{\int_0^L k(x)T_h'(x)w_i'(x)\,\mathrm dx=
\int_0^L Q(x)w_i(x)\,\mathrm dx+\left[kT_h'w_i\right]_0^L}.$$

Теперь нужны только первые производные $T_h$ и $w_i$, а не вторые. Именно поэтому можно использовать непрерывные функции из прямых отрезков. Их производные на элементах постоянны; значения производных в отдельных узлах не влияют на интеграл. Разрыв $k(x)$ учитывается интегралами по двум материалам; отдельно дифференцировать $k$ не требуется.

**Уточнение:** для кусочно-линейного $T_h$ сильная невязка формально понимается в обобщённом смысле. Для реализации используем непосредственно слабую форму; вторую производную шапочки вычислять не требуется.

### Граничные условия в слабой форме

Граничный член раскрывается так:
$$[kT'w]_0^L=k(L)T'(L)w(L)-k(0)T'(0)w(0).$$

**Дирихле:** температура на граничном узле задана, $T=T_0$. Тестовая функция в этом узле равна нулю, поэтому соответствующий граничный член исчезает. В системе уравнений граничную строку заменяем уравнением $T_i=T_0$.

**Неймана:** на границе задано $kT'=q$. Значение входит в правую часть: справа добавляется $+q_R$, слева - $-q_L$. Для теплоизолированного конца $q_R=0$ граничный член естественным образом исчезает; дополнительная строка с производной не нужна.

**Важно о знаке:** $q=kT'$ - условное обозначение, а физический поток вдоль оси $x$ равен $\Phi=-q$. Это не одно и то же. В коде ненулевое правое условие Неймана всегда означает именно $k_RT'(L)=q_R$.

## 4. Аналитическое решение при кусочно-постоянной теплопроводности

На каждой половине уравнение сводится к $kT''+Q=0$. Дважды интегрируя, получаем
$$T(x)=-\frac{Q}{2k}x^2+C_1x+C_2.$$
Для общего решения нужно выполнить граничные условия и непрерывность температуры и потока на границе материалов:
$$T^\alpha(x)=\begin{cases}
-\dfrac{Q}{2k_L}x^2+C_1^{\alpha,L}x+C_2^{\alpha,L},&0\le x<L/2,\\
-\dfrac{Q}{2k_R}(x-L)^2+C_1^{\alpha,R}(x-L)+C_2^{\alpha,R},&L/2\le x\le L,
\end{cases}$$
$$T^\alpha(L/2-0)=T^\alpha(L/2+0),\qquad
k_LT'^\alpha(L/2-0)=k_RT'^\alpha(L/2+0).$$
Индекс $\alpha$ обозначает вариант граничных условий. Обозначения $L/2-0$ и $L/2+0$ означают подход к границе слева и справа. Разные наклоны температуры допустимы: непрерывен поток $kT'$, а не обязательно сама производная.

Для двух температур: $C_2^{D,L}=T_{0L}$, $C_2^{D,R}=T_{0R}$. Для температуры слева и потока справа: $C_2^{N,L}=T_{0L}$, $C_1^{N,R}=q_R/k_R$.

### 4.1. Две заданные температуры: DD

Подставим условия сшивки при $x=L/2$. На правой половине здесь $x-L=-L/2$, поэтому
$$\begin{cases}
-\dfrac{QL^2}{8k_L}+C_1^{D,L}\dfrac L2+T_{0L}
=-\dfrac{QL^2}{8k_R}-C_1^{D,R}\dfrac L2+T_{0R},\\
-\dfrac{QL}{2}+k_LC_1^{D,L}=\dfrac{QL}{2}+k_RC_1^{D,R}.
\end{cases}$$
Из второго уравнения:
$$k_LC_1^{D,L}=QL+k_RC_1^{D,R},\qquad
C_1^{D,R}=\frac{k_LC_1^{D,L}-QL}{k_R}.$$
Подставляя в первое и собирая слагаемые, получаем
$$\frac L2\left(1+\frac{k_L}{k_R}\right)C_1^{D,L}
=T_{0R}-T_{0L}+\frac{QL^2}{8}\left(\frac1{k_L}+\frac3{k_R}\right).$$
Обозначим $\Delta T=T_{0R}-T_{0L}$. Тогда
$$\boxed{C_1^{D,L}=\frac{2k_R\Delta T}{L(k_L+k_R)}+
\frac{QL(k_R+3k_L)}{4k_L(k_L+k_R)}},$$
$$\boxed{C_1^{D,R}=\frac{2k_L\Delta T}{L(k_L+k_R)}-
\frac{QL(k_L+3k_R)}{4k_R(k_L+k_R)}}.$$
Следовательно,
$$\boxed{T^D(x)=\begin{cases}
-\dfrac{Qx^2}{2k_L}+C_1^{D,L}x+T_{0L},&0\le x<L/2,\\
-\dfrac{Q(x-L)^2}{2k_R}+C_1^{D,R}(x-L)+T_{0R},&L/2\le x\le L.
\end{cases}}$$

Обратите внимание: при сшивке на правой половине $x-L=-L/2$. В знаменателях констант стоит сумма $k_L+k_R$, поэтому решение корректно и при $k_L=k_R$.

### Удобная эквивалентная запись для программы

Интегрируя исходное уравнение один раз, получаем $kT'=C-Qx$. Введём
$$A_0=\frac L2\left(\frac1{k_L}+\frac1{k_R}\right),\qquad
A_1=\frac{L^2}{8}\left(\frac1{k_L}+\frac3{k_R}\right).$$
Из $\Delta T=CA_0-QA_1$ находим
$$C=\frac{\Delta T+QA_1}{A_0},\qquad C_1^{D,L}=\frac C{k_L},\qquad C_1^{D,R}=\frac{C-QL}{k_R}.$$
Эта запись даёт те же константы, но содержит меньше отдельных слагаемых.

### 4.2. Температура слева и поток справа: DN

Пусть $T(0)=T_{0L}$, $k_RT'(L)=q_R$. Тогда
$$C_2^{N,L}=T_{0L},\qquad C_1^{N,R}=\frac{q_R}{k_R}.$$
Условия сшивки:
$$\begin{cases}
-\dfrac{QL^2}{8k_L}+C_1^{N,L}\dfrac L2+T_{0L}
=-\dfrac{QL^2}{8k_R}-C_1^{N,R}\dfrac L2+C_2^{N,R},\\
-\dfrac{QL}{2}+k_LC_1^{N,L}=\dfrac{QL}{2}+k_RC_1^{N,R}.
\end{cases}$$
Из второго уравнения:
$$k_LC_1^{N,L}=QL+q_R,\qquad \boxed{C_1^{N,L}=\frac{QL+q_R}{k_L}}.$$
Из первого:
$$\boxed{C_2^{N,R}=T_{0L}+\frac{QL^2}{8}\left(\frac1{k_R}-\frac1{k_L}\right)
+\frac{(QL+q_R)L}{2k_L}+\frac{q_RL}{2k_R}}.$$
Таким образом,
$$\boxed{T^N(x)=\begin{cases}
-\dfrac{Qx^2}{2k_L}+\dfrac{QL+q_R}{k_L}x+T_{0L},&0\le x<L/2,\\
-\dfrac{Q(x-L)^2}{2k_R}+\dfrac{q_R}{k_R}(x-L)+C_2^{N,R},&L/2\le x\le L.
\end{cases}}$$
Для изолированного конца $q_R=0$:
$$C_1^{N,L}=\frac{QL}{k_L},\quad C_1^{N,R}=0,\quad
C_2^{N,R}=T_{0L}+\frac{QL^2}{8}\left(\frac1{k_R}-\frac1{k_L}\right)+\frac{QL^2}{2k_L}.$$

**Проверки на бумаге:** подставьте $x=0,L,L/2$; продифференцируйте обе ветви; проверьте уравнение и поток. При $k_L=k_R=k$ формулы упрощаются до
$$T^D=T_{0L}+\frac{\Delta T}{L}x+\frac{Q}{2k}x(L-x),$$
$$T^N=T_{0L}+\frac{QL+q_R}{k}x-\frac{Q}{2k}x^2.$$
При $I=0$ исчезает источник $Q$: в однородном материале DD даёт прямую, а DN при $q_R=0$ - постоянную температуру.

## 5. Численная постановка: только треугольные функции

Разделим стержень узлами $0=x_0<x_1<\dots<x_N=L$. Элемент $e$ - отрезок $[x_e,x_{e+1}]$, его длина $h_e=x_{e+1}-x_e$. Узел $L/2$ обязательно включаем в сетку: тогда внутри каждого элемента $k_e$ постоянен. Строим отдельно левую и правую половины, объединяя их без повторения середины.

### Локальные линейные функции

На эталонном отрезке $[-1,1]$ линейные функции имеют вид
$$\hat\phi_0(\eta)=\frac{1-\eta}{2},\qquad\hat\phi_1(\eta)=\frac{1+\eta}{2},\qquad
\hat\phi'_0=-\frac12,\quad\hat\phi'_1=\frac12.$$
Для программы удобнее $\xi=(x-x_e)/h_e\in[0,1]$, где $\eta=2\xi-1$. Тогда
$$\hat\phi_0=1-\xi,\quad\hat\phi_1=\xi,\qquad
\frac{\mathrm d\hat\phi_0}{\mathrm dx}=-\frac1{h_e},\quad
\frac{\mathrm d\hat\phi_1}{\mathrm dx}=\frac1{h_e}.$$
Не перепутайте производные по $\xi$ и по физической координате $x$.

### Глобальная шапочка

Для внутреннего узла $i$:
$$\phi_i(x)=\begin{cases}
(x-x_{i-1})/(x_i-x_{i-1}),&x_{i-1}\le x\le x_i,\\
(x_{i+1}-x)/(x_{i+1}-x_i),&x_i<x\le x_{i+1},\\
0,&\text{на остальных элементах}.
\end{cases}$$
У крайнего узла остаётся только одна половина шапочки. В своём узле $\phi_i=1$, во всех остальных узлах $\phi_i=0$. На левом соседнем элементе это локальная правая функция, а на правом - локальная левая. Так две прямые образуют треугольник.

Приближение:
$$T_h(x)=\sum_{j=0}^{N}T_j\phi_j(x),\qquad w_i=\phi_i.$$
Неизвестные $T_j$ - температуры в узлах. Между узлами температура интерполируется линейно. Сумма шапочек в любой точке стержня равна единице.

### Локальные интегралы

Подставляя разложение в слабую форму, получаем $KT=f$, где
$$K_{ij}=\int_0^L k\phi_j'\phi_i'\,\mathrm dx,\qquad f_i=\int_0^L Q\phi_i\,\mathrm dx+\text{граничный вклад}.$$
На элементе производные постоянны, поэтому
$$\boxed{K_e=\frac{k_e}{h_e}\begin{pmatrix}1&-1\\-1&1\end{pmatrix}},\qquad
\boxed{f_e=\frac{Qh_e}{2}\begin{pmatrix}1\\1\end{pmatrix}}.$$
Для нагрузки использовано $\int_0^1(1-\xi)\,\mathrm d\xi=\int_0^1\xi\,\mathrm d\xi=1/2$. Численная квадратура здесь не требуется.

### Сборка на примере двух элементов

Для узлов $0,L/2,L$ положим $a=k_L/(L/2)$, $b=k_R/(L/2)$. До применения граничных условий:
$$K=\begin{pmatrix}a&-a&0\\-a&a+b&-b\\0&-b&b\end{pmatrix},\qquad
f=\frac{QL}{4}\begin{pmatrix}1\\2\\1\end{pmatrix}.$$
Средний узел получает вклад обоих элементов. Локальные индексы $0,1$ элемента $e$ соответствуют глобальным $e,e+1$. В матрицу и вектор вклады **прибавляются**, а не записываются поверх соседних вкладов.

### Применение граничных условий и алгоритм

Для учебной реализации сначала соберите всю систему. Для условия Дирихле обнулите только соответствующую строку, поставьте единицу на диагональ и заданную температуру в правую часть. Столбец оставьте: его коэффициенты по-прежнему учитывают заданную температуру в соседних уравнениях. После замены строк матрица может стать несимметричной - это нормально для выбранного способа. Не обнуляйте столбцы без переноса их вкладов в правую часть.

Слева всегда $T(0)=T_{0L}$. Справа либо такая же замена строки для $T(L)=T_{0R}$, либо прибавление $q_R$ к последней компоненте $f$ без замены строки.

Алгоритм: построить сетку -> вычислить $Q$ -> для каждого элемента получить $k_e,K_e,f_e$ -> собрать $K,f$ -> применить условия -> решить `np.linalg.solve(K, f)` -> интерполировать -> сравнить с аналитикой. Порядок базиса фиксирован; ни спектральных функций, ни выбора порядка нет.

## 6. Подготовка Python

Нужны только NumPy и Matplotlib. Ячейка настройки готова. Число элементов в каждой половине - положительное целое; всего элементов вдвое больше.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.5), "font.size": 11,
    "axes.grid": True, "figure.dpi": 110,
})

L = 0.5                 # длина стержня, м
S = 1.0e-6              # м^2
rho = 1.2e-6            # Ом*м
I = 0.1                 # А
k_left, k_right = 5.0, 10.0  # Вт/(м*К)
T_left, T_right = 0.0, 100.0 # град. C
q_right = 0.0           # k*T'(L), Вт/м^2; ноль означает изоляцию
n_half = 10             # элементов в каждой половине
x_plot = np.linspace(0.0, L, 401)

### 6.1. Источник тепла

Функция определяет, сколько тепла электрический ток выделяет в единице объёма стержня за единицу времени. Используйте закон Джоуля и формулу из раздела 2. На вход подаются ток, удельное сопротивление и площадь сечения в СИ; результатом должно быть одно число Q в Вт/м^3.

**Пример для проверки:** Проверка: `heat_source(0.1, 1.2e-6, 1e-6)` -> `12000.0`; при нулевом токе -> `0.0`.

In [2]:
def heat_source(current, resistivity, area):
    """current - ток (А), resistivity - сопротивление (Ом*м), area - площадь (м^2). Результат: Q (Вт/м^3), float."""
    return (current**2 * resistivity / area**2)


### 6.2. Сетка

Функция разбивает стержень на небольшие отрезки и возвращает координаты их концов. В каждой половине должно быть `n_half` равных элементов. Середина стержня обязательно является узлом, поскольку в ней меняется материал; в итоговом массиве она должна встречаться только один раз.

**Пример для проверки:** При `length=1.0, n_half=2` результат `[0, 0.25, 0.5, 0.75, 1]`.

In [6]:
def make_mesh(length, n_half):
    """Результат: возрастающий массив узлов формы (2*n_half+1,), содержащий length/2."""
    return np.linspace(0,length,n_half*2+1)

### 6.3. Теплопроводность

Функция сообщает, из какого материала состоит стержень в заданных точках, возвращая соответствующую теплопроводность. На левой половине используйте `k_left`, на правой - `k_right`; точку в середине отнесите к правой половине. Функция должна работать как с одной координатой, так и с массивом координат.

**Пример для проверки:** Проверка для `length=1`: в точках `[0.25, 0.5, 0.75]` получаем `[5, 10, 10]`.

In [10]:
def conductivity(x, length, k_left, k_right):
    """x - координата (м); length - длина (м); k_left, k_right - теплопроводности. Результат: число или массив."""
    return k_left if x<length/2 else k_right


### 6.4. Базисная шапочка

Функция вычисляет значения треугольной шапочки, связанной с узлом `i`. В этом узле она равна единице, к соседним узлам линейно уменьшается до нуля и далее остаётся нулевой. У крайних узлов есть только одна половина шапочки. Результат показывает, какой вклад температура этого узла вносит в приближение в точках `x`.

**Пример для проверки:** Для `nodes=[0, 0.5, 1]`, `i=1`, `x=[0, 0.25, 0.5, 0.75, 1]` ожидается `[0, 0.5, 1, 0.5, 0]`.

In [ ]:
def hat_function(x, nodes, i):
    """x - скаляр или массив; nodes - узлы; i - индекс от 0 до len(nodes)-1. Результат той же формы, что x."""
    nodes = np.asarray(nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    n = len(nodes) - 1
    val = np.zeros_like(x)

    if i > 0:
        m = (x >= nodes[i-1]) & (x <= nodes[i])
        val[m] = (x[m] - nodes[i-1]) / (nodes[i] - nodes[i-1])
    if i < n:
        m = (x >= nodes[i]) & (x <= nodes[i+1])
        val[m] = (nodes[i+1] - x[m]) / (nodes[i+1] - nodes[i])

    return val



### 6.5. Производная шапочки

Функция вычисляет наклон шапочки узла `i` в заданных точках. На возрастающей ветви наклон положительный, на убывающей - отрицательный; его величина зависит от длины соответствующего элемента. Вне шапочки возвращайте ноль. В самих узлах также примем значение ноль, поскольку отдельные точки не влияют на интегралы.

**Пример для проверки:** Для предыдущей сетки в точках `[0.25, 0.75]` производная средней шапочки `[2, -2]`.

In [2]:
def hat_derivative(x, nodes, i):
    """Производная по физической координате x (1/м), с той же формой, что x; в узлах - 0."""
    dfdx = 1/(nodes[1]-nodes[0]) #для равномерной сетки!!!
    a = (i>0) & (x<nodes[i] and x>nodes[i-1])
    b = (i<len(nodes)-1) & (x<nodes[i+1] and x>nodes[i])
    X = np.zeros_like(x)
    X[a] +=dfdx
    X[b] -=dfdx 
    return X


### 6.6. Локальная матрица

Функция описывает связь температур на двух концах одного элемента при теплопроводности `k_value` и длине `h`. Верните матрицу размера 2 x 2 по формуле раздела 5. Она учитывает перенос тепла внутри элемента; вклады соседних элементов будут объединены позже.

**Пример для проверки:** Для `h=0.5, k_value=2` ожидается `[[4, -4], [-4, 4]]`. Суммы строк нулевые, матрица симметрична.

In [8]:
def element_matrix(h, k_value):
    """h - длина элемента (м), k_value - его теплопроводность. Результат: массив (2, 2)."""
    return k_value/h * np.array([[1,-1],[-1,1]])

### 6.7. Локальная нагрузка

Функция определяет вклад внутреннего нагрева одного элемента в уравнения для двух его концов. При постоянном источнике этот вклад распределяется между концами поровну. Верните вектор из двух чисел по формуле раздела 5; их сумма должна соответствовать источнику, умноженному на длину элемента.

**Пример для проверки:** Для `h=0.5, source=4` ожидается `[1, 1]`; сумма двух компонент равна `source*h`.

In [5]:
def element_load(h, source):
    """source - постоянный Q (Вт/м^3), h - длина элемента. Результат: массив (2,)."""
    return h*source/2*np.ones((2))

### 6.8. Глобальная сборка

Функция объединяет все элементы в одну систему уравнений для температур стержня. Для каждого элемента используйте его длину и теплопроводность в середине, получая локальные матрицу и нагрузку через предыдущие функции. Вклады элементов, которые имеют общий узел, должны складываться. Верните общую матрицу K и вектор f, пока без граничных условий.

**Пример для проверки:** На узлах `[0, 0.5, 1]`, при обоих коэффициентах `2` и `source=4`, ожидается матрица `[[4,-4,0],[-4,8,-4],[0,-4,4]]` и вектор `[1,2,1]`. Граничные условия здесь ещё не применяются.

In [ ]:
def assemble_system(nodes, length, k_left, k_right, source):
    """Результат: пара (K, f), формы (M, M) и (M,), M=len(nodes)."""
        

### 6.9. Граничные условия

Функция добавляет к собранной системе условия на концах стержня. Слева температура задана всегда; справа строка `"D"` означает заданную температуру, а `"N"` - заданную величину q=k*T'(L). Для заданной температуры соответствующая строка становится уравнением для этого узла; заданный правый поток добавляется в нагрузку. Верните изменённые копии матрицы и вектора, сохранив исходные массивы.

**Пример для проверки:** Проверка: при `right_type="N", right_value=3` последняя нагрузка увеличивается на 3; первая строка становится `[1,0,...]`, её нагрузка - `left_temperature`.

In [ ]:
def apply_boundary_conditions(K, f, left_temperature, right_type, right_value):
    """Результат: пара (K_bc, f_bc) тех же форм."""
    pass

### 6.10. Решение МКЭ

Функция связывает основные этапы расчёта: сборку системы, добавление граничных условий и решение линейных уравнений. Используйте `assemble_system`, `apply_boundary_conditions` и готовый решатель `np.linalg.solve`. Результатом должен быть массив температур в узлах, расположенных в том же порядке, что и в `nodes`.

**Пример для проверки:** Проверка: на узлах `[0,0.5,1]`, при `k_left=k_right=2`, `source=0`, температурах `0,100` получите `[0,50,100]`.

In [ ]:
def solve_fem(nodes, length, k_left, k_right, source, left_temperature, right_type, right_value):
    """Результат: массив узловых температур (len(nodes),), град. C."""
    pass

### 6.11. Восстановление температуры

Функция восстанавливает температуру в произвольных точках внутри стержня по уже найденным узловым значениям. Между соседними узлами температура изменяется линейно. Можно использовать сумму шапочек с узловыми температурами или линейную интерполяцию. Верните температуры в точках `x`; в самих узлах результат должен совпадать с исходными узловыми значениями.

**Пример для проверки:** Для узлов `[0,0.5,1]` и температур `[0,50,100]` в точках `[0.25,0.75]` получите `[25,75]`.

In [ ]:
def interpolate_temperature(x, nodes, temperatures):
    """Результат: температура той же формы, что x; крайние точки включены."""
    pass

### 6.12. Аналитическая температура

Функция вычисляет температуру непосредственно по аналитическому решению раздела 4, независимо от численного расчёта. Выбирайте формулы по граничному условию справа: `"D"` для температуры, `"N"` для потока. На каждой половине стержня используйте соответствующую ветвь с её теплопроводностью. Верните значения в точках `x`; обе ветви должны давать одинаковую температуру в середине.

**Пример для проверки:** На `length=1`, `k_left=k_right=2`, `source=4`, слева 0: для DD с температурой справа 0 значение в середине `0.25`; для DN с q=0 значение справа `1.0`, с q=2 - `2.0`.

In [ ]:
def analytical_temperature(x, length, k_left, k_right, source, left_temperature, right_type, right_value):
    """x - скаляр/массив в [0,length]. Результат той же формы."""
    pass

### 6.13. Ошибка

Функция показывает наибольшее расхождение между численным и аналитическим решениями в выбранных точках. Сравните значения в одинаковых координатах и верните максимальную абсолютную разность одним числом. Для исследования точности сравнивайте решения не только в узлах, но и между ними.

**Пример для проверки:** Для `[1,3]` и `[1.5,2]` результат `1.0`.

In [ ]:
def max_error(numerical, exact):
    """Результат: float, максимальная абсолютная ошибка температуры (град. C)."""
    pass

## 7. Готовые опыты и графики

Расчётные функции выше пока пустые. Ячейки ниже показывают, как ими пользоваться. Графики появятся после заполнения нужных функций; отсутствие результата сейчас ожидаемо.

In [ ]:
source = heat_source(I, rho, S)
nodes = make_mesh(L, n_half)
if source is None or nodes is None:
    print("Заполните heat_source и make_mesh. Расчётные опыты пока пропущены.")
else:
    print(f"Источник: {source:g} Вт/м^3; элементов: {len(nodes)-1}; длина: {L:g} м.")

### 7.1. Вид шапочек и их производных

Почему сумма шапочек равна единице? Как меняется наклон при уменьшении длины элемента?

In [ ]:
# Небольшая учебная сетка не зависит от заполнения make_mesh.
basis_nodes = np.array([0.0, L/4, L/2, 3*L/4, L])
basis_values = [hat_function(x_plot, basis_nodes, i) for i in range(len(basis_nodes))]
basis_slopes = [hat_derivative(x_plot, basis_nodes, i) for i in range(len(basis_nodes))]
if any(v is None for v in basis_values + basis_slopes):
    print("Для рисунка заполните hat_function и hat_derivative.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for i, (values, slopes) in enumerate(zip(basis_values, basis_slopes)):
        axes[0].plot(x_plot, values, label=f"Узел {i}")
        axes[1].plot(x_plot, slopes, label=f"Узел {i}")
    axes[0].set(title="Треугольные базисные функции", xlabel="x, м", ylabel="phi_i(x)")
    axes[1].set(title="Производные по x", xlabel="x, м", ylabel="phi_i'(x), 1/м")
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
    plt.show()

### 7.2. Два варианта граничных условий

Сопоставьте численную ломаную и аналитическую кривую. Почему наклон может изменяться в середине? Где температура максимальна? Обязательно ли при внутреннем нагреве максимум находится на конце?

In [ ]:
solutions = {}
if source is None or nodes is None:
    print("Сначала заполните функции источника и сетки.")
else:
    for label, right_type, right_value in [
        ("DD: лёд и кипящая вода", "D", T_right),
        ("DN: лёд и изоляция", "N", q_right),
    ]:
        temperatures = solve_fem(nodes, L, k_left, k_right, source, T_left, right_type, right_value)
        exact = analytical_temperature(x_plot, L, k_left, k_right, source, T_left, right_type, right_value)
        numerical = None if temperatures is None else interpolate_temperature(x_plot, nodes, temperatures)
        if numerical is None or exact is None:
            print(f"{label}: заполните solve_fem, его зависимости, interpolate_temperature и analytical_temperature.")
            continue
        solutions[right_type] = (numerical, exact)
        error = max_error(numerical, exact)
        fig, ax = plt.subplots()
        ax.plot(x_plot, exact, label="Аналитическое решение", color="black")
        ax.plot(x_plot, numerical, "--", label="МКЭ")
        ax.plot(nodes, temperatures, "o", markersize=4, label="Узловые температуры")
        ax.axvline(L/2, color="gray", linestyle=":", label="Граница материалов")
        ax.set(title=label, xlabel="x, м", ylabel="Температура, град. C")
        ax.legend()
        plt.tight_layout()
        plt.show()
        if error is None:
            print("Заполните max_error для оценки ошибки.")
        else:
            print(f"Максимальная ошибка на выбранных точках: {error:.6g} град. C")

### 7.3. Сгущение сетки

Сравним сетки с 2, 4, 8, 16 элементами на каждую половину. Измеряем ошибку на общей подробной сетке, а не только в узлах: для данной кусочно-постоянной модели значения МКЭ в узлах могут совпадать с аналитикой почти до машинной точности, хотя между ними есть ошибка интерполяции. Для линейных элементов ожидается уменьшение максимальной ошибки примерно в четыре раза при уменьшении шага вдвое.

Почему проверка только в узлах вводит в заблуждение? Что произойдёт, если точка смены материала попадёт внутрь элемента?

In [ ]:
if source is None:
    print("Сначала заполните heat_source.")
else:
    refinement_steps, refinement_errors = [], []
    exact_refinement = analytical_temperature(x_plot, L, k_left, k_right, source, T_left, "D", T_right)
    for count in (2, 4, 8, 16):
        refined_nodes = make_mesh(L, count)
        if refined_nodes is None or exact_refinement is None:
            break
        refined_t = solve_fem(refined_nodes, L, k_left, k_right, source, T_left, "D", T_right)
        if refined_t is None:
            break
        refined_values = interpolate_temperature(x_plot, refined_nodes, refined_t)
        if refined_values is None:
            break
        refined_error = max_error(refined_values, exact_refinement)
        if refined_error is None:
            break
        refinement_steps.append(np.max(np.diff(refined_nodes)))
        refinement_errors.append(refined_error)
    if len(refinement_errors) != 4:
        print("Для исследования сетки заполните make_mesh, solve_fem, интерполяцию, аналитику и ошибку.")
    elif not np.all(np.asarray(refinement_errors) > 0):
        print("Есть нулевые ошибки: логарифмический график неприменим. Проверьте источник и точки сравнения.")
    else:
        fig, ax = plt.subplots()
        ax.loglog(refinement_steps, refinement_errors, "o-", label="Ошибка МКЭ")
        reference = refinement_errors[0] * (np.asarray(refinement_steps)/refinement_steps[0])**2
        ax.loglog(refinement_steps, reference, "--", label="Ориентир: ошибка пропорциональна h^2")
        ax.set(title="Ошибка при сгущении сетки (DD)", xlabel="Максимальный шаг h, м", ylabel="Максимальная ошибка, град. C")
        ax.legend()
        plt.tight_layout()
        plt.show()

### 7.4. Параметрическое исследование

Проведём два опыта для DD: изменим ток, затем теплопроводность правой половины, сохраняя остальные параметры. Сравнение DD и DN уже предусмотрено выше. После основных опытов можно изменить длину, температуры на концах и ненулевое $q_R$ для DN.

Как нагрев зависит от $I$ и от $I^2$? Что меняется, если поменять знак тока? Почему увеличение теплопроводности помогает отводу тепла? Как скачок коэффициента влияет на наклоны температуры? Является ли максимум температуры монотонной функцией выбранного параметра в ваших опытах?

Диапазоны ниже - небольшие учебные вариации. Если получаются чрезмерные температуры, обсудите, какие предположения модели перестают быть реалистичными.

In [ ]:
if nodes is None:
    print("Сначала заполните make_mesh.")
else:
    parameter_curves = {"Ток": [], "Правая теплопроводность": []}
    for current in (0.0, 0.05, 0.1):
        varied_source = heat_source(current, rho, S)
        if varied_source is None:
            break
        varied_t = solve_fem(nodes, L, k_left, k_right, varied_source, T_left, "D", T_right)
        varied_values = None if varied_t is None else interpolate_temperature(x_plot, nodes, varied_t)
        if varied_values is None:
            break
        parameter_curves["Ток"].append((f"I = {current:g} А", varied_values))
    if source is not None:
        for varied_k in (5.0, 10.0, 20.0):
            varied_t = solve_fem(nodes, L, k_left, varied_k, source, T_left, "D", T_right)
            varied_values = None if varied_t is None else interpolate_temperature(x_plot, nodes, varied_t)
            if varied_values is None:
                break
            parameter_curves["Правая теплопроводность"].append((f"k справа = {varied_k:g} Вт/(м*К)", varied_values))
    if any(len(curves) != 3 for curves in parameter_curves.values()):
        print("Для параметрических опытов заполните источник, решатель МКЭ и интерполяцию.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        for ax, (title, curves) in zip(axes, parameter_curves.items()):
            for label, values in curves:
                ax.plot(x_plot, values, label=label)
            ax.set(title=f"Влияние параметра: {title}", xlabel="x, м", ylabel="Температура, град. C")
            ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

## 8. Проверки и самостоятельные выводы

После реализации выполните контрольные примеры возле каждой функции. Затем проверьте:

- **Нулевой ток:** при одинаковом $k$ решение DD линейно; DN с изоляцией постоянно.
- **Одинаковые материалы:** аналитическая функция не делит на $k_L-k_R$; обе формулы совпадают с однородными решениями раздела 4.
- **Граница материалов:** температура непрерывна; аналитические значения $kT'$ совпадают слева и справа. Наклоны конечных элементов дают приближённый поток, поэтому их равенство в середине при конечном шаге не требуйте буквально.
- **Изоляция:** аналитически $T'(L)=0$. Наклон последнего линейного элемента приближается к нулю при сгущении сетки.
- **Ненулевое условие Неймана:** при $L=1$, $k_L=k_R=2$, $Q=4$, $T_{0L}=0$, $q_R=2$ должно быть $T(L)=2$ и $kT'(L)=2$.
- **Баланс:** $\Phi(L)-\Phi(0)=QL$. При $q_R=0$ всё выделенное тепло уходит через левый конец.
- **Сборка:** до граничных условий матрица симметрична, суммы строк равны нулю, сумма нагрузки равна $QL$; после условий система имеет единственное решение.

### Ваш отчёт

Запишите ответы в следующую ячейку: модель и допущения; алгоритм; результаты проверки; сравнение с аналитическим решением; влияние параметров и граничных условий. Ссылайтесь на собственные графики и указывайте параметры каждого опыта. Не заменяйте выводы пересказом формул.

### Мои выводы

1. Модель и её допущения: ...
2. Последовательность алгоритма: ...
3. Сравнение с аналитикой и проверка сетки: ...
4. Влияние тока, теплопроводности и условий на концах: ...
5. Ограничения модели и дополнительные вопросы: ...

## Дополнительная литература

Слабая форма и граничные условия рассмотрены в учебнике: [The FEniCS Tutorial, Combining Dirichlet and Neumann conditions](https://hplgit.github.io/fenics-tutorial/pub/html/._ftut1010.html). В источнике поток определён через внешнюю нормаль; в этом блокноте используется явно указанное обозначение $q=kT'$ вдоль оси $x$, поэтому знаки сопоставляются с учётом этого определения.